# Notebook 3: Agentic RAG Introduction

In this notebook you will learn how to classify a user search and have specialized agents called depending on the user's question. This notebook does not use the Azure AI Search index to create a full solution for you - you will get to do that in the hands on exercise.

## Learning Objectives
- Learn about the WorkflowBuilder and conditional routing using switch-case logic
- Implement a classifier to recognize the user's intent and route to a specialized agent
- Implement agents to respond to specific search types


### Install Required Packages

In [4]:
#r "nuget:Microsoft.Agents.AI, *-*"
#r "nuget:Microsoft.Agents.AI.Workflows, *-*"
#r "nuget:Microsoft.Agents.AI.OpenAI, *-*"
#r "nuget:Microsoft.Extensions.AI, *-*"
#r "nuget:Microsoft.Extensions.AI.OpenAI, *-*"
#r "nuget:Azure.AI.OpenAI, *-*"
#r "nuget:Azure.Core, *-*"
#r "nuget:Azure.Identity, *-*"
#r "nuget:Azure.Search.Documents, *-*"
#r "nuget:System.Linq.AsyncEnumerable, *-*"
#r "nuget:Microsoft.Extensions.Configuration, 10.0.6"
#r "nuget:Microsoft.Extensions.Configuration.Json, 10.0.6"
#r "nuget:Microsoft.Extensions.Configuration.Binder, 10.0.6"
#r "nuget:Microsoft.Extensions.Configuration.EnvironmentVariables, 10.0.6"

Installed Packages Azure.AI.OpenAI, 2.9.0-beta.1 Azure.Core, 1.53.0 Azure.Identity, 1.21.0 Azure.Search.Documents, 11.8.0-beta.1 microsoft.agents.ai, 1.1.0 Microsoft.Agents.AI.OpenAI, 1.1.0 Microsoft.Agents.AI.Workflows, 1.1.0 Microsoft.Extensions.AI, 10.5.0 Microsoft.Extensions.AI.OpenAI, 10.5.0 Microsoft.Extensions.Configuration, 10.0.6 Microsoft.Extensions.configuration.Binder, 10.0.6 Microsoft.Extensions.Configuration.EnvironmentVariables, 10.0.6 Microsoft.Extensions.Configuration.Json, 10.0.6 System.Linq.AsyncEnumerable, 11.0.0-preview.3.26207.106

### Setup the Module Imports

In [5]:
using Microsoft.Extensions.Configuration;
using Microsoft.Extensions.AI;
using System.IO;
using Azure.AI.OpenAI;
using Azure.Identity;
using OpenAI.Embeddings;
using System.Text.Json;
using System.Collections.Generic;
using System.Linq;
using System.Threading.Tasks;
using Azure.Search.Documents;
using Azure.Search.Documents.Models;
using System.ComponentModel;
using Microsoft.Agents.AI;
using Microsoft.Agents.AI.Workflows;
using OpenAI.Chat;


### Get the needed environment variables

In [6]:
public const string DefaultConfigFileName = "appsettings.Local.json";

public static string FindConfigDirectory(string fileName)
{
    var directory = new DirectoryInfo(Directory.GetCurrentDirectory());

    while (directory is not null)
    {
        if (File.Exists(Path.Combine(directory.FullName, fileName)))
        {
            return directory.FullName;
        }
        directory = directory.Parent;
    }

    return null;
}

var basePath = FindConfigDirectory(DefaultConfigFileName)
            ?? throw new InvalidOperationException(
                $"Could not find {DefaultConfigFileName} in current directory or any parent directory.");

// Load configuration from appsettings.json
var configuration = new ConfigurationBuilder()
    .SetBasePath(basePath)
    .AddJsonFile("appsettings.Local.json", optional: true, reloadOnChange: true) // Optional environment-specific settings
    .AddEnvironmentVariables()
    .Build();


foreach (var kvp in configuration.AsEnumerable())
{
    if (!string.IsNullOrEmpty(kvp.Value))
    {
        Environment.SetEnvironmentVariable(kvp.Key, kvp.Value);
    }
}

var searchSettings = new 
{
    Endpoint = Environment.GetEnvironmentVariable("AZURE_SEARCH_ENDPOINT") ?? string.Empty,
    ApiKey = Environment.GetEnvironmentVariable("AZURE_SEARCH_API_KEY") ?? string.Empty,
    IndexName = Environment.GetEnvironmentVariable("AZURE_SEARCH_INDEX_NAME") ?? string.Empty
};

var openAISettings = new
{
    Endpoint = Environment.GetEnvironmentVariable("AZURE_OPENAI_ENDPOINT") ?? string.Empty,
    ApiVersion = Environment.GetEnvironmentVariable("AZURE_OPENAI_API_VERSION") ?? string.Empty,
    ChatDeploymentName = Environment.GetEnvironmentVariable("AZURE_OPENAI_CHAT_DEPLOYMENT_NAME") ?? string.Empty,
    EmbeddingDeploymentName = Environment.GetEnvironmentVariable("AZURE_OPENAI_EMBEDDING_DEPLOYMENT") ?? string.Empty,
};


Define the AI functions you'll use for testing the specific search type in this notebook.

> NOTE: these are where the actual implementation goes for doing specific search types in the hands on exercise

In [7]:
public class SearchTools
{
    [Description("Answers yes or no questions.")]
    public string YesOrNoSearch([Description("User question to answer with yes or no")] string userQuestion)
    {
        return $"Question: {userQuestion}\nAnswer: NO";
    }

    [Description("Answers questinos that required counting.")]
    public string CountSearch([Description("User question requiring counting items")] string userQuestion)
    {
        return $"Question: {userQuestion}\nAnswer: 42";
    }

    [Description("Answers simple question that requires semantic search.")]
    public string SemanticSearch([Description("User question requiring semantic search")]string userQuestion)
    {
        return $"Question: {userQuestion}\nAnswer: They all the gray or metallic in color.";
    }
}

Create a prompt for the classifier to be able to determine how to route the user's question.

In [8]:
var CLASSIFIER_AGENT_INSTRUCTIONS = """
   You are a query classification system for an IT support ticket database.
   Classify the incoming user question into exactly one category and return
   a JSON object with "category" and "reasoning" fields.

   ## Database Schema
   The database contains IT support tickets with these fields:
   - Id: unique identifier
   - Create_Date: date the ticket was created
   - Subject: ticket subject
   - Body: ticket question/description
   - Answer: ticket response/solution
   - Type: ticket type (values: "Incident", "Request", "Problem", "Change")
   - Queue: department name (values: "Human Resources", "IT", "Finance", "Operations", "Sales", "Marketing", "Engineering", "Support")
   - Priority: priority level (values: "high", "medium", "low")
   - Language: ticket language
   - Business_Type: business category
   - Tags: categorization tags

   **IMPORTANT**: When "and" combines field values (Type, Queue, Priority), these are FILTERS for counting/searching, NOT separate items to compare.

   ## Categories (use these exact values):

   **difference**: Questions with negation/exclusion ("not", "don't", "without", "excluding").
      - "Which Dell XPS Issue does not mention Windows?" -> difference
      - "Find tickets without high priority" -> difference

   **intersection**: Questions combining multiple SEARCH TOPICS with "and", "both", "that also".
      - "What issues are for Dell XPS laptops and the user tried Win + Ctrl + Shift + B?" -> intersection
      - NOT when "and" combines field filters like Priority/Queue/Type.

   **multi_hop**: Questions asking for a different attribute than what's searched (find X, extract Y).
      - "What department had consultants with Login Issues?" -> multi_hop
      - "Which queue handles password reset requests?" -> multi_hop

   **comparative**: Questions comparing multiple items ("more", "less", "vs", "or").
      - "Do we have more issues with MacBook Air or Dell XPS?" -> comparative

   **yes_no**: Explicit yes/no questions expecting a boolean answer.
      - "Are there any issues for Dell XPS laptops?" -> yes_no

   **count**: Counting questions ("how many", "count", "total", "number of").
      - "How many Incidents for Human Resources and low priority?" -> count (all filters!)

   **semantic_search**: General queries about issues, solutions, how-to (everything else).
      - "What problems are there with Surface devices?" -> semantic_search

   ## Classification Priority
   1. COUNTING first: "how many", "count", "total" -> count
   2. NEGATION: "not", "don't", "without" -> difference
   3. COMPARISON: "more", "less", "vs", "or" comparing items -> comparative
   4. INTERSECTION: multiple search topics with "and" (NOT field filters) -> intersection
   5. MULTI-HOP: "What [FIELD] had [CONDITION]" -> multi_hop
   6. YES/NO: explicit boolean questions -> yes_no
   7. Everything else -> semantic_search

   ## Key Rules
   - Field values (Priority, Queue, Type) are FILTERS, not search topics
   - "How many X and Y and Z?" = count (filters). "What X and Y?" = intersection (topics)
   - "Which X does not mention Y?" = difference, NOT count
""";

### Define the Data Models and Workflow Executors

This cell defines the core building blocks for the classifier workflow:

**Data Models**
- `ClassifyResult` — the JSON shape the classifier LLM returns (`category` + `reasoning`)
- `ClassifiedQuery` — carries the classified category alongside the original user question through the workflow

**Executors** (the units of work inside a `Workflow`)
- `ClassifierExecutor` — an `Executor<ChatMessage, ClassifiedQuery>` that sends the user message to the classifier agent, parses the structured JSON response (stripping any markdown fences the model may wrap around it), and returns a `ClassifiedQuery`
- `SpecialistExecutor` — a reusable `Executor<ClassifiedQuery>` decorated with `[YieldsOutput]`. It forwards the user question to whichever specialist agent it wraps and yields that agent's response as the final workflow output

**Routing Helper**
- `CategoryConditions.Is(category)` — returns a predicate `Func<object?, bool>` that checks whether a `ClassifiedQuery` matches the given category. These predicates are used by the switch-case routing in the workflow builder

In [17]:
using System.Text.Json;
using System.Text.Json.Serialization;
using System.Threading;
using Microsoft.Agents.AI;
using Microsoft.Agents.AI.Workflows;
using Microsoft.Extensions.AI;

public sealed class ClassifyResult
{
    [JsonPropertyName("category")]
    public string Category { get; set; } = "";

    [JsonPropertyName("reasoning")]
    public string Reasoning { get; set; } = "";
}

public sealed class ClassifiedQuery
{
    public string Category { get; set; } = "";
    public string UserQuestion { get; set; } = "";
}

internal sealed class ClassifierExecutor : Executor<Microsoft.Extensions.AI.ChatMessage, ClassifiedQuery>
{
    private readonly AIAgent _classifierAgent;

    public ClassifierExecutor(AIAgent classifierAgent) : base("ClassifierExecutor")
    {
        _classifierAgent = classifierAgent;
    }

    public override async ValueTask<ClassifiedQuery> HandleAsync(
        Microsoft.Extensions.AI.ChatMessage message,
        IWorkflowContext context,
        CancellationToken cancellationToken = default)
    {
        var response = await _classifierAgent.RunAsync(message, cancellationToken: cancellationToken);
        var text = response.Text?.Trim() ?? "";

        // Strip markdown code fences if present
        if (text.StartsWith("```"))
        {
            var firstNewline = text.IndexOf('\n');
            if (firstNewline >= 0)
            {
                text = text[(firstNewline + 1)..];
            }
            var lastFence = text.LastIndexOf("```");
            if (lastFence >= 0)
            {
                text = text[..lastFence].Trim();
            }
        }

        // Extract the first complete JSON object using brace-depth counting
        int braceDepth = 0;
        int jsonEnd = -1;
        for (int i = 0; i < text.Length; i++)
        {
            if (text[i] == '{') braceDepth++;
            else if (text[i] == '}')
            {
                braceDepth--;
                if (braceDepth == 0)
                {
                    jsonEnd = i + 1;
                    break;
                }
            }
        }
        if (jsonEnd > 0)
        {
            text = text[..jsonEnd];
        }

        var result = JsonSerializer.Deserialize<ClassifyResult>(text, new JsonSerializerOptions
        {
            PropertyNameCaseInsensitive = true
        }) ?? new ClassifyResult { Category = "semantic_search", Reasoning = "Failed to parse classification" };

        Console.WriteLine($"  [Classified as: {result.Category} -- {result.Reasoning}]");

        return new ClassifiedQuery
        {
            Category = result.Category,
            UserQuestion = message.Text ?? ""
        };
    }
}

[YieldsOutput(typeof(string))]
internal sealed class SpecialistExecutor : Executor<ClassifiedQuery>
{
    private readonly AIAgent _agent;

    public SpecialistExecutor(string name, AIAgent agent) : base(name)
    {
        _agent = agent;
    }

    public override async ValueTask HandleAsync(
        ClassifiedQuery query,
        IWorkflowContext context,
        CancellationToken cancellationToken = default)
    {
        Console.WriteLine($"\n  [{_agent.Name ?? Id}] processing: {query.UserQuestion}");

        var response = await _agent.RunAsync(query.UserQuestion, cancellationToken: cancellationToken);

        Console.WriteLine();
        await context.YieldOutputAsync(response.Text ?? "No response generated.", cancellationToken);
    }
}

public static class CategoryConditions
{
    public static Func<object?, bool> Is(string category) =>
        result => result is ClassifiedQuery q &&
                  string.Equals(q.Category, category, StringComparison.OrdinalIgnoreCase);
}




(117,30): warning CS8632: The annotation for nullable reference types should only be used in code within a '#nullable' annotations context.

warning CS1701: Assuming assembly reference 'Microsoft.Extensions.AI.Abstractions, Version=10.4.0.0, Culture=neutral, PublicKeyToken=31bf3856ad364e35' used by 'Microsoft.Agents.AI.Abstractions' matches identity 'Microsoft.Extensions.AI.Abstractions, Version=10.5.0.0, Culture=neutral, PublicKeyToken=31bf3856ad364e35' of 'Microsoft.Extensions.AI.Abstractions', you may need to supply runtime policy

warning CS1701: Assuming assembly reference 'Microsoft.Extensions.AI.Abstractions, Version=10.4.0.0, Culture=neutral, PublicKeyToken=31bf3856ad364e35' used by 'Microsoft.Agents.AI.Abstractions' matches identity 'Microsoft.Extensions.AI.Abstractions, Version=10.5.0.0, Culture=neutral, PublicKeyToken=31bf3856ad364e35' of 'Microsoft.Extensions.AI.Abstractions', you may need to supply runtime policy



### Create the Agents

Define a factory method that creates all four agents from a single `ChatClient`:

- **Classifier agent** — uses `ChatClientAgentOptions` with `ResponseFormat = ForJsonSchema<ClassifyResult>()` so the model is constrained to return valid JSON matching our `ClassifyResult` schema (structured output)
- **Specialist agents** (`yes_no`, `count`, `semantic_search`) — each is created with `AsAIAgent()`, given a focused system instruction, and wired to its corresponding `SearchTools` method via `AIFunctionFactory.Create()` so the agent can call that tool

In [18]:

(AIAgent classifierAgent, AIAgent yesNoAgent, AIAgent countAgent, AIAgent semanticSearchAgent)
    CreateAgents(ChatClient chatClient)
{
    // Classifier agent: Acts as the search planner and dispatcher
    var classifierAgent = chatClient.AsAIAgent(new ChatClientAgentOptions
    {
        ChatOptions = new()
        {
            Instructions = CLASSIFIER_AGENT_INSTRUCTIONS,
            ResponseFormat = Microsoft.Extensions.AI.ChatResponseFormat.ForJsonSchema<ClassifyResult>()
        }
    });

    var searchTools = new SearchTools();

    // Yes/No search specialist: Handles yes/no questions
    var yesNoAgent = chatClient.AsAIAgent(
        instructions: "You handle yes/no questions.",
        name: "yes_no_agent",
        tools: [AIFunctionFactory.Create(searchTools.YesOrNoSearch)]
    );

    // Count search specialist: Handles counting questions
    var countAgent = chatClient.AsAIAgent(
        instructions: "You handle questions that require counting items.",
        name: "count_agent",
        tools: [AIFunctionFactory.Create(searchTools.CountSearch)]
    );

    // Semantic search specialist: Handles semantic search questions
    var semanticSearchAgent = chatClient.AsAIAgent(
        instructions: "You handle questions that require semantic search.",
        name: "semantic_search_agent",
        tools: [AIFunctionFactory.Create(searchTools.SemanticSearch)]
    );

    return (classifierAgent, yesNoAgent, countAgent, semanticSearchAgent);
}


warning CS1701: Assuming assembly reference 'Microsoft.Extensions.AI.Abstractions, Version=10.4.0.0, Culture=neutral, PublicKeyToken=31bf3856ad364e35' used by 'Microsoft.Agents.AI' matches identity 'Microsoft.Extensions.AI.Abstractions, Version=10.5.0.0, Culture=neutral, PublicKeyToken=31bf3856ad364e35' of 'Microsoft.Extensions.AI.Abstractions', you may need to supply runtime policy

warning CS1701: Assuming assembly reference 'Microsoft.Extensions.AI.Abstractions, Version=10.4.0.0, Culture=neutral, PublicKeyToken=31bf3856ad364e35' used by 'Microsoft.Agents.AI' matches identity 'Microsoft.Extensions.AI.Abstractions, Version=10.5.0.0, Culture=neutral, PublicKeyToken=31bf3856ad364e35' of 'Microsoft.Extensions.AI.Abstractions', you may need to supply runtime policy

warning CS1701: Assuming assembly reference 'OpenAI, Version=2.9.1.0, Culture=neutral, PublicKeyToken=b4187f3e65366280' used by 'Microsoft.Agents.AI.OpenAI' matches identity 'OpenAI, Version=2.10.0.0, Culture=neutral, PublicK

### Build the Workflow with Switch-Case Routing

This method uses `WorkflowBuilder` to wire the classifier and specialists into a single executable workflow:

1. **Entry point** — the builder is initialized with `classifierExecutor`, which runs first on every incoming message
2. **`AddSwitch`** — creates conditional routing based on the classifier's output. Each `AddCase` maps a category string (via a `CategoryConditions.Is()` predicate) to a specialist executor:
   - `"yes_no"` routes to `yesNoExecutor`
   - `"count"` routes to `countExecutor`
   - `WithDefault` catches all other categories and routes to `semanticSearchExecutor`
3. **`WithOutputFrom`** — declares which executors produce the final workflow output (all three specialists in this case)

In [21]:
using Microsoft.Agents.AI.Workflows;

static Workflow BuildWorkflow(Dictionary<string, AIAgent> agents)
{
    // Create executors
    var classifierExecutor = new ClassifierExecutor(agents["classifier"]);
    var yesNoExecutor = new SpecialistExecutor("YesNo", agents["yes_no"]);
    var semanticSearchExecutor = new SpecialistExecutor("SemanticSearch", agents["semantic_search"]);
    var countExecutor = new SpecialistExecutor("Count", agents["count"]);
    

    // Build workflow with switch-case routing
    var builder = new WorkflowBuilder(classifierExecutor);
    builder.AddSwitch(classifierExecutor, sb => sb
        .AddCase(CategoryConditions.Is("yes_no"), yesNoExecutor)
        .AddCase(CategoryConditions.Is("count"), countExecutor)
        .WithDefault(semanticSearchExecutor)
    )
    .WithOutputFrom(yesNoExecutor, countExecutor, semanticSearchExecutor);

    return builder.Build();
}

### Wire Everything Together

Now all the code is defined, you get to use it. The cell below:

1. Creates an `AzureOpenAIClient` using `DefaultAzureCredential` and gets a `ChatClient` for the configured deployment
2. Calls `CreateAgents` to build the classifier and three specialist agents
3. Stores them in a `Dictionary<string, AIAgent>` keyed by role name so `BuildWorkflow` can look them up
4. Calls `BuildWorkflow` to assemble the complete routing workflow

In [25]:
var chatClient = new AzureOpenAIClient(
            new Uri(openAISettings.Endpoint),
            new DefaultAzureCredential()
         )
        .GetChatClient(openAISettings.ChatDeploymentName);

// Create all agents: classifier + specialists
var (classifierAgent, yesNoAgent, countAgent, semanticSearchAgent) = CreateAgents(chatClient);

var agents = new Dictionary<string, AIAgent>
{
    ["classifier"] = classifierAgent,
    ["yes_no"] = yesNoAgent,
    ["count"] = countAgent,
    ["semantic_search"] = semanticSearchAgent
};

var workflow = BuildWorkflow(agents);


warning CS1701: Assuming assembly reference 'OpenAI, Version=2.9.1.0, Culture=neutral, PublicKeyToken=b4187f3e65366280' used by 'Azure.AI.OpenAI' matches identity 'OpenAI, Version=2.10.0.0, Culture=neutral, PublicKeyToken=b4187f3e65366280' of 'OpenAI', you may need to supply runtime policy

warning CS1701: Assuming assembly reference 'System.ClientModel, Version=1.9.0.0, Culture=neutral, PublicKeyToken=92742159e12e44c8' used by 'Azure.AI.OpenAI' matches identity 'System.ClientModel, Version=1.10.0.0, Culture=neutral, PublicKeyToken=92742159e12e44c8' of 'System.ClientModel', you may need to supply runtime policy

warning CS1701: Assuming assembly reference 'Azure.Core, Version=1.51.1.0, Culture=neutral, PublicKeyToken=92742159e12e44c8' used by 'Azure.AI.OpenAI' matches identity 'Azure.Core, Version=1.53.0.0, Culture=neutral, PublicKeyToken=92742159e12e44c8' of 'Azure.Core', you may need to supply runtime policy

warning CS1701: Assuming assembly reference 'OpenAI, Version=2.9.1.0, Cult

For demo purposes create a list of some sample questions.

In [26]:
var demo_questions = new List<string>() {
    "How many tickets were logged and Incidents for Human Resources and low priority?", // count search
    "Are there any issues logged for Dell XPS laptops?" // yes/no search
};

### Run the Workflow

The `RunWorkflowAsync` helper defined above does the following:
- `InProcessExecution.RunStreamingAsync` kicks off the workflow with a `ChatMessage` containing the user's question
- `WatchStreamAsync` yields `WorkflowEvent` objects as the workflow progresses; we filter for `WorkflowOutputEvent` to print the final answer from whichever specialist handled the query

Now loop through the demo questions to see if the classifier correctly routes each one to the right specialist agent.

In [29]:
static async Task RunWorkflowAsync(Workflow workflow, string question)
{
    await using StreamingRun run = await InProcessExecution.RunStreamingAsync(
        workflow, new Microsoft.Extensions.AI.ChatMessage(ChatRole.User, question));

    await foreach (WorkflowEvent evt in run.WatchStreamAsync())
    {
        if (evt is WorkflowOutputEvent output)
        {
            Console.WriteLine(output.Data);
        }
    }
}

In [33]:
for (int i = 0; i < demo_questions.Count; i++)
{
    Console.WriteLine($"\n--- Query {i + 1}/{demo_questions.Count} ---");
    Console.WriteLine($"User: {demo_questions[i]}");
    await RunWorkflowAsync(workflow, demo_questions[i]);
    Console.WriteLine();
}


--- Query 1/2 ---
User: How many tickets were logged and Incidents for Human Resources and low priority?
User: How many tickets were logged and Incidents for Human Resources and low priority?
  [Classified as: count -- The question asks 'How many tickets', which indicates counting. The query includes multiple filters for Type (Incidents), Queue (Human Resources), and Priority (low), which aligns with counting tickets meeting specified criteria.]

  [count_agent] processing: How many tickets were logged and Incidents for Human Resources and low priority?

There were 42 tickets and incidents logged for Human Resources with low priority.


--- Query 2/2 ---
User: Are there any issues logged for Dell XPS laptops?
  [Classified as: yes_no -- The question explicitly asks for a yes/no answer about the existence of any issues logged for Dell XPS laptops, which matches the yes_no category.]

  [yes_no_agent] processing: Are there any issues logged for Dell XPS laptops?

No, there are no issues

As I've pointed out in the other notebooks, the question "How many tickets were logged and Incidents for Human Resources and low priority?" is a hard one to get the system to answer. This time the classifier correctly identified it as something the count_agent should take care of - which means we can now handle special retreival to answer it. That is a start!

Next is the hands on exercise where you get to turn these lessons from the notebooks into an Agentic RAG solution for the IT support search index we've been using.